# CLERNet — Model Execution

Runs a pre-trained CLERNet model on a set of test instances and writes
a **predictions JSON** that the `Aggregation-Selection` notebook consumes.

For each instance the JSON stores:
- `predictions`: raw per-fluent sigmoid outputs at every observed step
- `possible_goals_indices`: candidate goal hypotheses from `hyps.dat`
- `correct_goal_indices` / `correct_hyp_idx`: ground truth


## 1. Configuration


In [ ]:
import os

REPO_ROOT   = os.path.abspath(os.path.join(os.getcwd(), ".."))  # CLERNet_public/
MODELS_DIR  = os.path.join(REPO_ROOT, "models")
DATA_DIR    = os.path.join(REPO_ROOT, "data", "custom_dataset")
OUTPUT_DIR  = os.path.join(REPO_ROOT, "results")

DOMAIN      = "blocksworld"  # blocksworld | depots | driverlog | logistics | satellite | zenotravel
N_INSTANCES = 10          # set to None to run all available instances


## 2. Imports


In [18]:
import sys
sys.path.insert(0, REPO_ROOT)

import json
import zipfile
import numpy as np
from os.path import join

from src.constants import MAX_PLAN_DIMS, FILENAMES
from src.models.architecture import CUSTOM_OBJECTS
from src.utils import load_from_pickles
from src.cli.test_model import _load_model, _parse_obs, _parse_real_hyp, _encode_actions


## 3. Load Model and Vocabularies


In [19]:
max_plan_dim = MAX_PLAN_DIMS[DOMAIN]
dict_dir     = join(DATA_DIR, DOMAIN)
model_path   = join(MODELS_DIR, f"{DOMAIN}.keras")

print(f"Loading vocabularies from {dict_dir} ...")
[action_vocab, goal_vocab] = load_from_pickles(
    dict_dir, [FILENAMES.ACTION_DICT_FILENAME, FILENAMES.GOALS_DICT_FILENAME]
)
print(f"  Actions : {len(action_vocab)}")
print(f"  Goals   : {len(goal_vocab)}")

print(f"\nLoading model from {model_path} ...")
model = _load_model(model_path, CUSTOM_OBJECTS)
model.summary()


Loading vocabularies from /home/lserina/OnlineGR/CLERNet_public/data/custom_dataset/blocksworld ...
dizionario loaded from /home/lserina/OnlineGR/CLERNet_public/data/custom_dataset/blocksworld
dizionario_goal loaded from /home/lserina/OnlineGR/CLERNet_public/data/custom_dataset/blocksworld
  Actions : 968
  Goals   : 506

Loading model from /home/lserina/OnlineGR/CLERNet_public/models/blocksworld.keras ...
Model: "blocksworld"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 75)]              0         
                                                                 
 embedding (Embedding)       (None, 75, 185)           179265    
                                                                 
 lstm_layer_0 (LSTM)         (None, 75, 461)           1193068   
                                                                 
 time_distributed (TimeDist  (None, 75, 5

## 4. Parsing Helpers

`parse_hyps` reads `hyps.dat` — each line is one candidate goal hypothesis with
comma-separated predicates — and converts each to a sorted list of goal vocab indices.

`find_correct_hyp_idx` identifies which hypothesis in `hyps.dat` matches `real_hyp.dat`.


In [20]:
def pred_to_idx(pred: str, goal_vocab: dict):
    """Lowercase predicate string → integer goal vocabulary index, or None."""
    if pred not in goal_vocab:
        return None
    return int(np.argmax(goal_vocab[pred]))


def parse_hyps(hyps_text: str, goal_vocab: dict) -> list:
    """Parse hyps.dat into a list of hypotheses.

    Each line is one hypothesis; predicates are comma-separated, e.g.:
        (ON A K), (ON B H), ...

    Returns:
        List of hypotheses, each a sorted list of goal vocabulary indices.
    """
    hypotheses = []
    for line in hyps_text.strip().splitlines():
        indices = []
        for token in line.split(","):
            pred = token.strip().strip("()").strip().lower()
            idx  = pred_to_idx(pred, goal_vocab)
            if idx is not None:
                indices.append(idx)
        if indices:
            hypotheses.append(sorted(indices))
    return hypotheses


def find_correct_hyp_idx(correct_indices: list, possible_goals: list) -> int:
    """Return the index of the hypothesis that matches the true goal, or -1."""
    correct_set = set(correct_indices)
    for i, hyp in enumerate(possible_goals):
        if set(hyp) == correct_set:
            return i
    return -1


## 5. Run Inference and Save Predictions

For each instance, the model predicts a per-fluent activation vector at every
observation step. Raw outputs (before any thresholding or aggregation) are stored
in the JSON so the `Aggregation-Selection` notebook can experiment with different
aggregation strategies.


In [21]:
zip_dir   = join(DATA_DIR, DOMAIN, "100")
zip_files = sorted(f for f in os.listdir(zip_dir) if f.endswith(".zip"))
if N_INSTANCES is not None:
    zip_files = zip_files[:N_INSTANCES]

os.makedirs(OUTPUT_DIR, exist_ok=True)
all_predictions = {}

for idx, zf in enumerate(zip_files, 1):
    zip_path = join(zip_dir, zf)
    with zipfile.ZipFile(zip_path) as z:
        obs_text      = z.read("obs.dat").decode()
        real_hyp_text = z.read("real_hyp.dat").decode()
        hyps_text     = z.read("hyps.dat").decode()

    actions         = _parse_obs(obs_text)
    goal_predicates = _parse_real_hyp(real_hyp_text)

    correct_indices = sorted(filter(None, [
        pred_to_idx(p, goal_vocab) for p in goal_predicates
    ]))
    possible_goals  = parse_hyps(hyps_text, goal_vocab)
    correct_hyp_idx = find_correct_hyp_idx(correct_indices, possible_goals)

    x      = _encode_actions(actions, action_vocab, max_plan_dim)
    y_pred = model.predict(x[np.newaxis, :], verbose=0)[0]  # (max_plan_dim, n_goals)

    obs_len = min(len(actions), max_plan_dim)

    all_predictions[zf] = {
        "obs_len":               obs_len,
        "correct_goal_indices":  correct_indices,
        "possible_goals_indices": possible_goals,
        "correct_hyp_idx":       correct_hyp_idx,
        "predictions":           y_pred[:obs_len].tolist(),  # shape: (obs_len, n_goals)
    }

    print(
        f"[{idx:2d}/{len(zip_files)}] {zf[:50]:<50}  "
        f"steps={obs_len}  n_hyps={len(possible_goals)}  "
        f"correct_hyp={correct_hyp_idx}"
    )

out_path = join(OUTPUT_DIR, f"{DOMAIN}_predictions.json")
with open(out_path, "w") as f:
    json.dump(all_predictions, f)
print(f"\nSaved {len(all_predictions)} instances → {out_path}")


[ 1/754] blocksworld_p000002_hyp=hyp-6_100.zip               steps=34  n_hyps=20  correct_hyp=6
[ 2/754] blocksworld_p000012_hyp=hyp-15_100.zip              steps=75  n_hyps=20  correct_hyp=15
[ 3/754] blocksworld_p000014_hyp=hyp-9_100.zip               steps=75  n_hyps=20  correct_hyp=9
[ 4/754] blocksworld_p000020_hyp=hyp-16_100.zip              steps=34  n_hyps=20  correct_hyp=16
[ 5/754] blocksworld_p000026_hyp=hyp-9_100.zip               steps=68  n_hyps=20  correct_hyp=9
[ 6/754] blocksworld_p000035_hyp=hyp-17_100.zip              steps=34  n_hyps=20  correct_hyp=17
[ 7/754] blocksworld_p000041_hyp=hyp-9_100.zip               steps=44  n_hyps=20  correct_hyp=9
[ 8/754] blocksworld_p000042_hyp=hyp-5_100.zip               steps=38  n_hyps=20  correct_hyp=5
[ 9/754] blocksworld_p000050_hyp=hyp-1_100.zip               steps=40  n_hyps=20  correct_hyp=1
[10/754] blocksworld_p000051_hyp=hyp-3_100.zip               steps=32  n_hyps=20  correct_hyp=3
[11/754] blocksworld_p000057_hyp=hyp-